# Semana 23 — Pricing avanzado y gestión de proyectos IA

**Objetivo:** construir las herramientas de negocio que usarás en cada proyecto:
calculadora de estimación PERT, generador de presupuestos en Markdown,
plantilla de change request y calculadora de coste de operación para el cliente.

**Al finalizar tendrás** en `herramientas-negocio/`:
- `estimador.py` — estimación PERT por componentes
- `presupuesto.py` — generador de presupuesto en Markdown
- `change_request.py` — generador de solicitudes de cambio
- `coste_operacion.py` — calculadora de coste mensual de APIs
- Ejemplos de uso con el proyecto ResearchAgent como caso real

**Coste estimado:** $0.00 — este notebook no llama a la OpenAI API.

## Celda 0 — Setup

In [ ]:
import os, json
from pathlib import Path
from datetime import date, timedelta
from dataclasses import dataclass, field
from typing import Optional

BASE_DIR = Path("../herramientas-negocio").resolve()
BASE_DIR.mkdir(exist_ok=True)

HOY = date.today()
print(f"✓ Directorio: {BASE_DIR}")
print(f"✓ Fecha: {HOY.isoformat()}")

## Celda 1 — Estimador PERT

Estima cada componente del proyecto con los tres escenarios y calcula el total.

In [ ]:
@dataclass
class ComponenteProyecto:
    nombre: str
    optimista: float   # días
    probable: float
    pesimista: float
    descripcion: str = ""

    @property
    def pert(self) -> float:
        """Estimación PERT: (O + 4P + Pe) / 6"""
        return (self.optimista + 4 * self.probable + self.pesimista) / 6

    @property
    def desviacion(self) -> float:
        """Desviación estándar PERT: (Pe - O) / 6"""
        return (self.pesimista - self.optimista) / 6


def estimar_proyecto(
    componentes: list[ComponenteProyecto],
    margen_pct: float = 0.20,
    day_rate: float = 400.0,
) -> dict:
    """
    Calcula la estimación total del proyecto con margen.
    Devuelve días, precio y desglose por componente.
    """
    total_pert  = sum(c.pert for c in componentes)
    total_opt   = sum(c.optimista for c in componentes)
    total_pes   = sum(c.pesimista for c in componentes)
    total_con_margen = total_pert * (1 + margen_pct)

    return {
        "dias_optimista":    round(total_opt, 1),
        "dias_probable":     round(total_pert, 1),
        "dias_pesimista":    round(total_pes, 1),
        "dias_con_margen":   round(total_con_margen, 1),
        "precio_minimo_eur": round(total_pert * day_rate, 0),
        "precio_objetivo_eur": round(total_con_margen * day_rate, 0),
        "margen_aplicado":   margen_pct,
        "day_rate":          day_rate,
        "desglose": [
            {
                "nombre":    c.nombre,
                "pert_dias": round(c.pert, 1),
                "pct_total": round(c.pert / total_pert * 100, 1),
                "rango":     f"{c.optimista}-{c.pesimista}d",
            }
            for c in componentes
        ],
    }


# ── Ejemplo: ResearchAgent para ANFACO-CYTMA ─────────────────────────────────
COMPONENTES_RESEARCH_AGENT = [
    ComponenteProyecto(
        "Setup e infraestructura",
        optimista=0.5, probable=1.0, pesimista=2.0,
        descripcion="FastAPI, Docker, Railway, CI/CD",
    ),
    ComponenteProyecto(
        "Ingeniería de prompts",
        optimista=1.0, probable=3.0, pesimista=6.0,
        descripcion="Planner, executor ReAct, synthesizer",
    ),
    ComponenteProyecto(
        "Integración corpus cliente",
        optimista=1.0, probable=3.0, pesimista=8.0,
        descripcion="Ingesta de documentos en Chroma, embedding",
    ),
    ComponenteProyecto(
        "Seguridad y guardrails",
        optimista=0.5, probable=1.5, pesimista=3.0,
        descripcion="Rate limiting, PII, moderación",
    ),
    ComponenteProyecto(
        "Observabilidad",
        optimista=0.5, probable=1.0, pesimista=2.0,
        descripcion="Tracer, Langfuse, logs estructurados",
    ),
    ComponenteProyecto(
        "Tests y evaluación",
        optimista=1.0, probable=2.0, pesimista=4.0,
        descripcion="Tests unitarios, integración, smoke",
    ),
    ComponenteProyecto(
        "Documentación y entrega",
        optimista=0.5, probable=1.0, pesimista=2.0,
        descripcion="README técnico, guía uso, resumen ejecutivo",
    ),
]

estimacion = estimar_proyecto(
    COMPONENTES_RESEARCH_AGENT,
    margen_pct=0.25,
    day_rate=450,
)

print("ESTIMACIÓN PERT — ResearchAgent")
print("=" * 55)
print(f"{'Componente':<30} {'PERT':>6} {'% total':>8} {'Rango':>10}")
print("-" * 55)
for c in estimacion["desglose"]:
    print(f"{c['nombre']:<30} {c['pert_dias']:>5}d {c['pct_total']:>7}% {c['rango']:>10}")
print("-" * 55)
print(f"{'TOTAL (sin margen)':<30} {estimacion['dias_probable']:>5}d")
print(f"{'TOTAL (con 25% margen)':<30} {estimacion['dias_con_margen']:>5}d")
print()
print("PRECIO:")
print(f"  Mínimo:   {estimacion['precio_minimo_eur']:,.0f}€  ({estimacion['dias_probable']}d × {estimacion['day_rate']}€/d)")
print(f"  Objetivo: {estimacion['precio_objetivo_eur']:,.0f}€  ({estimacion['dias_con_margen']}d × {estimacion['day_rate']}€/d)")
print()
print(f"  Rango a presentar al cliente: {estimacion['dias_optimista']}–{estimacion['dias_pesimista']} días")

## Celda 2 — Calculadora de coste de operación

Estima el coste mensual de APIs para el cliente en tres escenarios de uso.

In [ ]:
PRECIOS_OPENAI = {
    "gpt-4o-mini": {"input": 0.15e-6,  "output": 0.60e-6},
    "gpt-4o":      {"input": 2.50e-6,  "output": 10.0e-6},
    "text-embedding-3-small": {"input": 0.02e-6, "output": 0.0},
}


def estimar_coste_mensual(
    consultas_mes: int,
    tokens_input_media: int,
    tokens_output_media: int,
    modelo: str = "gpt-4o-mini",
    llamadas_por_consulta: int = 4,  # planner + pasos ReAct + synthesizer
) -> dict:
    p = PRECIOS_OPENAI[modelo]
    coste_llamada = (tokens_input_media * p["input"] +
                     tokens_output_media * p["output"])
    coste_consulta = coste_llamada * llamadas_por_consulta
    coste_mes      = coste_consulta * consultas_mes

    return {
        "modelo":              modelo,
        "consultas_mes":       consultas_mes,
        "llamadas_por_consulta": llamadas_por_consulta,
        "coste_por_consulta":  round(coste_consulta, 4),
        "coste_mes_eur":       round(coste_mes, 2),
        "coste_anio_eur":      round(coste_mes * 12, 2),
    }


# Tres escenarios para el cliente
ESCENARIOS = [
    {"nombre": "Uso bajo",    "consultas_mes": 50},
    {"nombre": "Uso esperado","consultas_mes": 200},
    {"nombre": "Uso alto",    "consultas_mes": 500},
]

# Parámetros del agente (de los logs de observabilidad de la semana 17)
TOKENS_INPUT_MEDIA  = 1_200  # por llamada
TOKENS_OUTPUT_MEDIA = 500

print("COSTE DE OPERACIÓN ESTIMADO — ResearchAgent")
print(f"Modelo: gpt-4o-mini | ~{TOKENS_INPUT_MEDIA} tokens input + {TOKENS_OUTPUT_MEDIA} output por llamada")
print(f"Llamadas por investigación: 4 (planner + 2 pasos ReAct + synthesizer)")
print()
print(f"{'Escenario':<16} {'Consultas/mes':>14} {'Coste/consulta':>15} {'Coste/mes':>11} {'Coste/año':>11}")
print("-" * 70)

for esc in ESCENARIOS:
    r = estimar_coste_mensual(
        consultas_mes=esc["consultas_mes"],
        tokens_input_media=TOKENS_INPUT_MEDIA,
        tokens_output_media=TOKENS_OUTPUT_MEDIA,
    )
    print(
        f"{esc['nombre']:<16} {r['consultas_mes']:>14} "
        f"{r['coste_por_consulta']:>14.4f}€ "
        f"{r['coste_mes_eur']:>10.2f}€ "
        f"{r['coste_anio_eur']:>10.2f}€"
    )

print()
print("Nota: estos costes van a cargo del cliente (API key propia).")
print("Incluye esta tabla en el contrato como 'Estimación de coste de operación'.")

## Celda 3 — Generador de presupuesto en Markdown

In [ ]:
@dataclass
class DatosPresupuesto:
    # Partes
    consultor_nombre: str
    consultor_email: str
    cliente_nombre: str
    cliente_contacto: str
    # Proyecto
    nombre_proyecto: str
    descripcion_breve: str
    # Estimación
    componentes: list[ComponenteProyecto]
    day_rate: float
    margen_pct: float = 0.25
    # Condiciones
    dias_validez: int = 15
    anticipo_pct: float = 0.40
    modelo_precio: str = "fijo"  # "fijo" | "tm" | "retainer"
    incluye: list[str] = field(default_factory=list)
    excluye: list[str] = field(default_factory=list)
    supuestos: list[str] = field(default_factory=list)


def generar_presupuesto_md(datos: DatosPresupuesto) -> str:
    est = estimar_proyecto(datos.componentes, datos.margen_pct, datos.day_rate)
    fecha_emision = HOY.isoformat()
    fecha_validez = (HOY + timedelta(days=datos.dias_validez)).isoformat()
    precio        = est["precio_objetivo_eur"]
    anticipo      = round(precio * datos.anticipo_pct, 0)
    resto         = precio - anticipo

    include_items = "\n".join(f"- {i}" for i in datos.incluye) or "- Ver desglose de componentes"
    exclude_items = "\n".join(f"- {i}" for i in datos.excluye) or "- (ninguna exclusión específica)"
    asump_items   = "\n".join(f"- {s}" for s in datos.supuestos) or "- (sin supuestos específicos)"

    desglose = "\n".join(
        f"| {c['nombre']:<30} | {c['rango']:>12} | {c['pert_dias']:>8}d | "
        f"{round(c['pert_dias'] * datos.day_rate):>10,}€ |"
        for c in est["desglose"]
    )

    return f"""# Presupuesto — {datos.nombre_proyecto}

| | |
|---|---|
| **Consultor** | {datos.consultor_nombre} — {datos.consultor_email} |
| **Cliente** | {datos.cliente_nombre} — {datos.cliente_contacto} |
| **Fecha de emisión** | {fecha_emision} |
| **Válido hasta** | {fecha_validez} |
| **Número de presupuesto** | PRE-{HOY.strftime('%Y%m')}-001 |

---

## Descripción del proyecto

{datos.descripcion_breve}

---

## Desglose de esfuerzo

| Componente | Rango (días) | Estimación | Importe |
|------------|:------------:|:----------:|--------:|
{desglose}
| **SUBTOTAL** | {est['dias_optimista']}–{est['dias_pesimista']}d | **{est['dias_probable']}d** | **{est['precio_minimo_eur']:,.0f}€** |
| **Margen de contingencia ({int(datos.margen_pct*100)}%)** | | | **{precio - est['precio_minimo_eur']:,.0f}€** |
| **TOTAL** | | **{est['dias_con_margen']}d** | **{precio:,.0f}€** |

*Day rate: {datos.day_rate:,.0f}€/día. IVA no incluido.*

---

## Condiciones de pago

| Hito | Importe |
|------|--------:|
| Anticipo al inicio ({int(datos.anticipo_pct*100)}%) | {anticipo:,.0f}€ |
| Entrega y aceptación ({int((1-datos.anticipo_pct)*100)}%) | {resto:,.0f}€ |
| **Total** | **{precio:,.0f}€** |

---

## Alcance

### Incluye
{include_items}

### No incluye
{exclude_items}

### Supuestos del proyecto
{asump_items}

Cualquier trabajo fuera de este alcance requerirá una solicitud de cambio
firmada por ambas partes antes de iniciar.

---

## Coste estimado de operación (a cargo del cliente)

| Escenario | Consultas/mes | Coste/mes |
|-----------|:-------------:|----------:|
| Uso bajo | 50 | ~0.70€ |
| Uso esperado | 200 | ~2.80€ |
| Uso alto | 500 | ~7.00€ |

*Coste de la API de OpenAI, con modelo gpt-4o-mini. El cliente gestiona su propia API key.*

---

*Para aceptar este presupuesto, responde a este email con confirmación escrita
o firma y devuelve este documento. El inicio del proyecto queda condicionado
al abono del anticipo.*
"""


# Genera el presupuesto del ResearchAgent
presupuesto_datos = DatosPresupuesto(
    consultor_nombre   = "Raffaele Bini",
    consultor_email    = "raffaele@dihdatalife.com",
    cliente_nombre     = "ANFACO-CYTMA",
    cliente_contacto   = "[contacto@anfaco.es]",
    nombre_proyecto    = "ResearchAgent — Agente de investigación automatizada",
    descripcion_breve  = (
        "Desarrollo de un agente de investigación autónomo que genera informes estructurados "
        "en español a partir de búsquedas web y la base documental interna del cliente. "
        "Incluye API REST, guardrails de seguridad, observabilidad y documentación completa."
    ),
    componentes        = COMPONENTES_RESEARCH_AGENT,
    day_rate           = 450,
    margen_pct         = 0.25,
    modelo_precio      = "fijo",
    incluye=[
        "Desarrollo completo del agente y la API REST",
        "Hasta 3 iteraciones de ajuste de prompts",
        "Tests automatizados (cobertura mínima 80%)",
        "Documentación técnica, guía de uso y resumen ejecutivo",
        "Despliegue en Railway (o el entorno acordado)",
        "2 semanas de soporte post-entrega (respuesta <24h)",
    ],
    excluye=[
        "Integración con sistemas no mencionados en este presupuesto",
        "Fine-tuning de modelos (requiere presupuesto separado)",
        "Coste mensual de la API de OpenAI (a cargo del cliente)",
        "Mantenimiento tras el período de soporte post-entrega",
        "Cambios de requisitos tras la aprobación del diseño",
    ],
    supuestos=[
        "El cliente proporciona acceso a sus documentos en los primeros 3 días laborables",
        "Hay una persona de contacto técnico disponible para responder dudas en <24h",
        "El entorno de producción tiene Python 3.11+ y acceso a internet",
        "Los datos de prueba no contienen información personal sensible de terceros",
    ],
)

presupuesto_md = generar_presupuesto_md(presupuesto_datos)
(BASE_DIR / "presupuesto_research_agent.md").write_text(presupuesto_md)
print("✓ presupuesto_research_agent.md generado")
print()
# Muestra el resumen de precio
est = estimar_proyecto(COMPONENTES_RESEARCH_AGENT, 0.25, 450)
print(f"Precio a presentar: {est['precio_objetivo_eur']:,.0f}€")
print(f"Anticipo (40%):     {est['precio_objetivo_eur']*0.4:,.0f}€")
print(f"Resto en entrega:   {est['precio_objetivo_eur']*0.6:,.0f}€")

## Celda 4 — Generador de change request

In [ ]:
@dataclass
class ChangeRequest:
    numero: int
    proyecto: str
    solicitado_por: str
    descripcion: str
    justificacion: str
    opciones: list[dict]  # [{"nombre", "dias", "precio_eur", "impacto_plazo"}]
    fecha: str = ""

    def __post_init__(self):
        if not self.fecha:
            self.fecha = HOY.isoformat()


def generar_change_request_md(cr: ChangeRequest) -> str:
    opciones_txt = ""
    for i, op in enumerate(cr.opciones, 1):
        opciones_txt += f"""
### Opción {i}: {op['nombre']}

- **Esfuerzo estimado:** {op['dias']} días
- **Coste adicional:** {op['precio_eur']:,.0f}€ + IVA
- **Impacto en plazo:** {op['impacto_plazo']}
- **Descripción:** {op.get('descripcion', '')}
"""

    return f"""# Solicitud de cambio #{cr.numero:03d}

| | |
|---|---|
| **Proyecto** | {cr.proyecto} |
| **Solicitado por** | {cr.solicitado_por} |
| **Fecha** | {cr.fecha} |
| **Estado** | ⏳ Pendiente de aprobación |

---

## Cambio solicitado

{cr.descripcion}

## Justificación

{cr.justificacion}

---

## Opciones propuestas
{opciones_txt}

---

## Decisión

- [ ] **Opción aprobada:** _______________
- [ ] **Rechazado** — Motivo: _______________
- [ ] **Aplazado a fase 2**

**Firma del cliente:** _______________ **Fecha:** _______________

**Firma del consultor:** _______________ **Fecha:** _______________

*Este documento forma parte del contrato del proyecto. Ningún trabajo adicional
comenzará sin la firma de ambas partes.*
"""


# Ejemplo real: el cliente pide añadir integración con EUMOFA
cr_ejemplo = ChangeRequest(
    numero=1,
    proyecto="ResearchAgent — ANFACO-CYTMA",
    solicitado_por="Director de Innovación, ANFACO-CYTMA",
    descripcion=(
        "Añadir soporte para consultar precios de mercado de la base de datos EUMOFA "
        "(European Market Observatory for Fisheries and Aquaculture) directamente desde "
        "el agente de investigación."
    ),
    justificacion=(
        "Los informes actuales no incluyen datos de precios de mercado, que son clave "
        "para los análisis de competitividad del sector conservero."
    ),
    opciones=[
        {
            "nombre": "Integración básica (scraping web EUMOFA)",
            "dias": 3,
            "precio_eur": 1350,
            "impacto_plazo": "+3 días laborables",
            "descripcion": "Scraping del portal público de EUMOFA. Frágil ante cambios en la web.",
        },
        {
            "nombre": "Integración API oficial EUMOFA",
            "dias": 5,
            "precio_eur": 2250,
            "impacto_plazo": "+1 semana",
            "descripcion": "Uso de la API REST oficial. Requiere registro y clave de acceso (gratuita).",
        },
        {
            "nombre": "Aplazar a fase 2",
            "dias": 0,
            "precio_eur": 0,
            "impacto_plazo": "Sin impacto en el proyecto actual",
            "descripcion": "Incluir en un sprint de mejoras tras la entrega. Presupuesto separado.",
        },
    ],
)

cr_md = generar_change_request_md(cr_ejemplo)
(BASE_DIR / "change_request_001.md").write_text(cr_md)
print("✓ change_request_001.md generado")
print()
print(cr_md[:800], "...")

## Celda 5 — Detector de señales de alerta en el proyecto

In [ ]:
@dataclass
class EstadoProyecto:
    dias_desde_inicio: int
    dias_estimados_total: float
    dias_trabajados: float
    pct_entregado: float       # 0.0 - 1.0
    dias_sin_datos_cliente: int
    num_cambios_requisitos: int
    reuniones_sin_decisor: int
    scope_creep_dias: float    # días de trabajo fuera de alcance ya realizados


def analizar_salud_proyecto(estado: EstadoProyecto) -> dict:
    alertas = []
    advertencias = []

    # Velocidad de avance
    velocidad_esperada = estado.dias_desde_inicio / estado.dias_estimados_total
    if estado.pct_entregado < velocidad_esperada * 0.7:
        alertas.append(
            f"Ritmo de entrega bajo: {estado.pct_entregado*100:.0f}% completado "
            f"con {velocidad_esperada*100:.0f}% del tiempo consumido."
        )

    # Bloqueos del cliente
    if estado.dias_sin_datos_cliente > 3:
        alertas.append(
            f"Bloqueo por cliente: {estado.dias_sin_datos_cliente} días sin recibir "
            f"datos o accesos necesarios. Notifica y documenta."
        )

    # Cambios de requisitos
    if estado.num_cambios_requisitos >= 2:
        alertas.append(
            f"{estado.num_cambios_requisitos} cambios de requisitos. "
            f"Cada cambio sin change request firmado es riesgo contractual."
        )
    elif estado.num_cambios_requisitos == 1:
        advertencias.append("1 cambio de requisitos. ¿Tienes change request firmado?")

    # Reuniones sin decisor
    if estado.reuniones_sin_decisor >= 2:
        advertencias.append(
            f"{estado.reuniones_sin_decisor} reuniones sin el decisor final. "
            f"Pide una reunión de alineación con quien aprueba."
        )

    # Scope creep
    pct_scope_creep = estado.scope_creep_dias / estado.dias_estimados_total
    if pct_scope_creep > 0.15:
        alertas.append(
            f"Scope creep: {estado.scope_creep_dias}d de trabajo fuera de alcance "
            f"({pct_scope_creep*100:.0f}% del total). Emite change request inmediatamente."
        )
    elif pct_scope_creep > 0.05:
        advertencias.append(
            f"{estado.scope_creep_dias}d de trabajo fuera de alcance "
            f"({pct_scope_creep*100:.0f}%). Monitoriza de cerca."
        )

    nivel = "🔴 CRÍTICO" if alertas else ("🟡 ATENCIÓN" if advertencias else "🟢 OK")

    return {
        "nivel":       nivel,
        "alertas":     alertas,
        "advertencias": advertencias,
        "resumen":     f"{len(alertas)} alerta(s), {len(advertencias)} advertencia(s)",
    }


# Ejemplo: proyecto con algunos problemas
ESCENARIOS_PROYECTO = [
    {
        "nombre": "Proyecto sano",
        "estado": EstadoProyecto(
            dias_desde_inicio=5, dias_estimados_total=14,
            dias_trabajados=5, pct_entregado=0.38,
            dias_sin_datos_cliente=0, num_cambios_requisitos=0,
            reuniones_sin_decisor=0, scope_creep_dias=0,
        )
    },
    {
        "nombre": "Proyecto con scope creep",
        "estado": EstadoProyecto(
            dias_desde_inicio=8, dias_estimados_total=14,
            dias_trabajados=8, pct_entregado=0.40,
            dias_sin_datos_cliente=2, num_cambios_requisitos=2,
            reuniones_sin_decisor=1, scope_creep_dias=2.5,
        )
    },
    {
        "nombre": "Proyecto en peligro",
        "estado": EstadoProyecto(
            dias_desde_inicio=12, dias_estimados_total=14,
            dias_trabajados=12, pct_entregado=0.50,
            dias_sin_datos_cliente=5, num_cambios_requisitos=3,
            reuniones_sin_decisor=3, scope_creep_dias=4,
        )
    },
]

for esc in ESCENARIOS_PROYECTO:
    resultado = analizar_salud_proyecto(esc["estado"])
    print(f"{'='*55}")
    print(f"{resultado['nivel']}  {esc['nombre']}")
    print(f"  {resultado['resumen']}")
    for a in resultado["alertas"]:
        print(f"  ⚠ ALERTA: {a}")
    for w in resultado["advertencias"]:
        print(f"  ℹ  AVISO:  {w}")
    print()

## Celda 6 — Guardado de herramientas como módulos reutilizables

In [ ]:
# Lista todos los entregables generados
print("Entregables generados en herramientas-negocio/:")
print()
for f in sorted(BASE_DIR.rglob("*.md")):
    rel = f.relative_to(BASE_DIR)
    lineas = len(f.read_text().splitlines())
    print(f"  {rel}  ({lineas} líneas)")

print()
print("Funciones disponibles en este notebook:")
funciones = [
    ("estimar_proyecto(componentes, margen, day_rate)",
     "Estimación PERT total con desglose por componente"),
    ("estimar_coste_mensual(consultas, tokens_in, tokens_out, modelo)",
     "Coste mensual de APIs para el cliente"),
    ("generar_presupuesto_md(DatosPresupuesto)",
     "Presupuesto completo en Markdown listo para enviar"),
    ("generar_change_request_md(ChangeRequest)",
     "Solicitud de cambio con opciones y firma"),
    ("analizar_salud_proyecto(EstadoProyecto)",
     "Detecta scope creep, bloqueos y alertas"),
]
for nombre, desc in funciones:
    print(f"  {nombre}")
    print(f"    → {desc}")
    print()

## Reto — Simulador de negociación de precio

El cliente te dice que el presupuesto que le enviaste está un 30% por encima de lo que tenía previsto.
En lugar de bajar el precio directamente, la técnica correcta es ajustar el alcance.

Implementa `proponer_alcance_reducido(componentes, presupuesto_cliente)` que:

1. Calcula cuántos días de trabajo caben en el presupuesto del cliente (con tu day rate)
2. Ordena los componentes por coste de mayor a menor
3. Propone dos versiones reducidas del proyecto:
   - **MVP**: elimina los componentes de menor prioridad hasta ajustarse al presupuesto
   - **Fase 1 / Fase 2**: divide los componentes en dos entregas, la primera dentro del presupuesto
4. Genera un Markdown con las dos propuestas para enviar al cliente

```python
def proponer_alcance_reducido(
    componentes: list[ComponenteProyecto],
    presupuesto_cliente: float,
    day_rate: float,
    prioridades: list[str],  # nombres de componentes en orden de prioridad
) -> str:
    # TODO
    pass

# Prueba: el cliente solo puede pagar 4.000€
propuesta = proponer_alcance_reducido(
    COMPONENTES_RESEARCH_AGENT,
    presupuesto_cliente=4000,
    day_rate=450,
    prioridades=[
        "Setup e infraestructura",
        "Ingeniería de prompts",
        "Tests y evaluación",
        "Integración corpus cliente",
        "Documentación y entrega",
        "Seguridad y guardrails",
        "Observabilidad",
    ],
)
print(propuesta)
```

## Resumen de la semana 23

| Herramienta | Dataclass / función | Cuándo usarla |
|------------|--------------------|--------------|
| Estimación PERT | `ComponenteProyecto` + `estimar_proyecto()` | Antes de dar cualquier precio |
| Coste de operación | `estimar_coste_mensual()` | En el contrato, sección 'costes a cargo del cliente' |
| Presupuesto | `DatosPresupuesto` + `generar_presupuesto_md()` | Para enviar al cliente antes de firmar |
| Change request | `ChangeRequest` + `generar_change_request_md()` | Cada vez que el cliente pide algo fuera de alcance |
| Salud del proyecto | `EstadoProyecto` + `analizar_salud_proyecto()` | Revisión semanal durante el proyecto |

La semana 24 es la **semana de cierre del plan**: revisión del portfolio, 
plan de acción para conseguir el primer cliente y hoja de ruta de los próximos seis meses.